# Hw5DiffusionTeam6 Colab Template

### 1) Mount Google Drive
Run this first to access dataset/checkpoint files stored in your Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2) Repository Setup
Set `REPO_URL` to your own repository before running the next cell.

In [2]:
# Example: set your repository coordinates, then authenticate via Colab Secrets / env / .env.local.
import os
import subprocess
from pathlib import Path

GITHUB_USERNAME = "MahikaGunjkar"
REPO_NAME = "Hw5DiffusionTeam6"
BRANCH = "main"


def _load_env_file(path: Path):
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and value and key not in os.environ:
            os.environ[key] = value


def _get_secret(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from google.colab import userdata
        value = (userdata.get(name) or "").strip()
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass
    return ""


repo_dir = Path("/content") / REPO_NAME
_load_env_file(repo_dir / ".env.local")
GITHUB_TOKEN = _get_secret("GITHUB_TOKEN")

repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git" if GITHUB_TOKEN else repo_url

if repo_dir.exists():
    print(f"Repo already exists at {repo_dir}; skipping clone.")
else:
    subprocess.run(["git", "clone", "-b", BRANCH, clone_url], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "remote", "set-url", "origin", repo_url], check=True)
    print(f"Cloned {repo_url} into {repo_dir}")

Cloning into 'Hw5DiffusionTeam6'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 116 (delta 52), reused 92 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 963.40 KiB | 4.68 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [3]:
# To pull latest changes safely (Must be in the repo dir, use pwd/ls to verify)
import os
import subprocess
from pathlib import Path

repo_dir = Path("/content") / REPO_NAME
assert repo_dir.exists(), f"Repo not found at {repo_dir}. Run the clone cell first."


def _load_env_file(path: Path):
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and value and key not in os.environ:
            os.environ[key] = value


def _get_secret(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from google.colab import userdata
        value = (userdata.get(name) or "").strip()
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass
    return ""


_load_env_file(repo_dir / ".env.local")
GITHUB_TOKEN = _get_secret("GITHUB_TOKEN")
repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
pull_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git" if GITHUB_TOKEN else repo_url

%cd /content/Hw5DiffusionTeam6
subprocess.run(["git", "pull", "--ff-only", pull_url, BRANCH], check=True)
subprocess.run(["git", "remote", "set-url", "origin", repo_url], check=True)
print("Repo updated with fast-forward pull.")

/content/Hw5DiffusionTeam6
From https://github.com/MahikaGunjkar/Hw5DiffusionTeam6
 * branch            main       -> FETCH_HEAD
HEAD is now at a3ff65f update yaml


In [4]:
%cd /content/Hw5DiffusionTeam6

/content/Hw5DiffusionTeam6


In [7]:
!pip install -r requirements.txt

In [8]:
from torch.utils.data import DataLoader
import yaml
import gc
import torch
from torchinfo import summary
import os
import json
import wandb
import pandas as pd
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### 2.1) Environment

In [9]:
import os

expected_repo = os.path.join('/content', REPO_NAME)
assert os.getcwd() == expected_repo, f"Current working directory is {os.getcwd()}, expected {expected_repo}."

required_paths = [
    'train.py',
    'configs/ddpm.yaml',
    'requirements.txt',
]
for p in required_paths:
    assert os.path.exists(p), f"Missing required file: {p}"

print('Repository check passed.')
print('cwd:', os.getcwd())


Repository check passed.
cwd: /content/Hw5DiffusionTeam6


### 3) Configure Paths

In [10]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/Project"
TAR_PATH = f"{DRIVE_ROOT}/imagenet100_128x128.tar.gz"
CKPT_SRC = f"{DRIVE_ROOT}/model.ckpt"
EXTRACT_ROOT = "/content/dataset_extracted"
OUTPUT_DIR = f"{DRIVE_ROOT}/experiments"

os.makedirs("pretrained", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXTRACT_ROOT, exist_ok=True)

assert os.path.exists(TAR_PATH), f"Dataset archive not found: {TAR_PATH}"
assert os.path.exists(CKPT_SRC), f"VAE checkpoint not found: {CKPT_SRC}"

print("TAR_PATH:", TAR_PATH)
print("CKPT_SRC:", CKPT_SRC)
print("OUTPUT_DIR:", OUTPUT_DIR)

TAR_PATH: /content/drive/MyDrive/Project/imagenet100_128x128.tar.gz
CKPT_SRC: /content/drive/MyDrive/Project/model.ckpt
OUTPUT_DIR: /content/drive/MyDrive/Project/experiments


### Copy VAE Checkpoint to local runtime
Copy `model.ckpt` to the fixed path expected by the training script: `pretrained/model.ckpt`.

In [11]:
import os

repo_dir = os.getcwd()
dst_dir = os.path.join(repo_dir, 'pretrained')
dst_ckpt = os.path.join(dst_dir, 'model.ckpt')
os.makedirs(dst_dir, exist_ok=True)

!cp "$CKPT_SRC" "$dst_ckpt"
assert os.path.exists(dst_ckpt), f"Failed to copy VAE checkpoint to {dst_ckpt}"
print(f"VAE checkpoint is ready: {dst_ckpt}")


VAE checkpoint is ready: /content/Hw5DiffusionTeam6/pretrained/model.ckpt


### 5) Extract Dataset and Locate Training Directory
If the archive is not extracted yet, extract it first, then automatically locate the `train/` directory.

In [12]:
import tarfile

if not os.path.exists(os.path.join(EXTRACT_ROOT, ".extracted")):
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        tar.extractall(EXTRACT_ROOT)
    with open(os.path.join(EXTRACT_ROOT, ".extracted"), "w", encoding="utf-8") as f:
        f.write("done")

candidate_roots = [
    os.path.join(EXTRACT_ROOT, "imagenet100_128x128"),
    os.path.join(EXTRACT_ROOT, "imagenet100"),
    EXTRACT_ROOT,
]
TRAIN_DIR = None
for root in candidate_roots:
    train_dir = os.path.join(root, "train")
    if os.path.isdir(train_dir):
        TRAIN_DIR = train_dir
        break

assert TRAIN_DIR is not None, "Could not find the train directory. Please verify the archive structure."
print("TRAIN_DIR:", TRAIN_DIR)

/tmp/ipykernel_3760/2443716922.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(EXTRACT_ROOT)


TRAIN_DIR: /content/dataset_extracted/imagenet100_128x128/train


### 6) Config

Write training settings to `configs/ddpm.yaml` (data path, `run_name`, batch size, etc.).

**Optional: W&B resume**

- In the code cell below, set `RESUME_ENABLED = True` and set `WANDB_CKPT_RUN_PATH` to the exact run you want to resume (`entity/project/run_id`). `train.py` treats this as the authoritative target for **both** checkpoint download and W&B log resume, and derives `wandb_run_id` from it automatically. **Leave `WANDB_CKPT_EPOCH` as `None`** to download the checkpoint artifact version with alias `latest`. Set `WANDB_CKPT_EPOCH` to an integer **only** when you need a specific `epoch-N` checkpoint artifact version.
- With `resume_enabled=true` and no local `--ckpt`, `train.py` downloads weights via **W&B Artifacts** into `wandb_ckpt_cache_dir` (default `./wandb_resume_cache`), then uses the existing `load_checkpoint` and output-dir logic. The target run must already have the expected checkpoint artifact version available. **An explicit `--ckpt` wins over W&B download.**
- **Auth:** downloads need `WANDB_API_KEY`. Prefer Colab **Secrets** with name `WANDB_API_KEY`; the next cell reads it from there. **Do not commit real keys to Git.**
- `WANDB_RESUME_POLICY` defaults to `must` in this repo because the intended behavior is: if the exact run cannot be resumed, fail fast instead of silently creating a new run. You can still override it to `allow` / `auto` / `never` when you explicitly want different behavior (see [Resuming a run](https://docs.wandb.ai/models/runs/resuming)). Keep `RESUME_ENABLED = False` for a fresh run; the trainer then ignores `wandb_run_id` / `wandb_resume` / `wandb_ckpt_*` in yaml to avoid accidental resume.


In [ ]:
from ruamel.yaml import YAML
import os

# ---- W&B resume overrides (must match configs/ddpm.yaml + train.py) ----
RESUME_ENABLED = False
WANDB_RESUME_POLICY = None  # None = let train.py default to "must"; else "allow" / "must" / "auto" / "never"
WANDB_RUN_ID = None  # optional override; when WANDB_CKPT_RUN_PATH is set, it must match that run id
WANDB_CKPT_RUN_PATH = None  # authoritative resume target, e.g. "your-entity/ddpm/abc123xyz"
WANDB_CKPT_EPOCH = None  # None = latest checkpoint artifact; else int N for alias epoch-N
WANDB_CKPT_CACHE_DIR = None  # None = yaml default; e.g. "/content/wandb_resume_cache"
# ---- W&B resume overrides (must match configs/ddpm.yaml + train.py) END----


yaml = YAML()
config_path = os.path.join(os.getcwd(), 'configs', 'ddpm.yaml')
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.load(f)

cfg['data_dir'] = TRAIN_DIR
cfg['output_dir'] = OUTPUT_DIR
cfg['run_name'] = 'cfg_baseline'
cfg['batch_size'] = 32
cfg['num_workers'] = 8

cfg['resume_enabled'] = RESUME_ENABLED
cfg['wandb_resume'] = WANDB_RESUME_POLICY
cfg['wandb_run_id'] = WANDB_RUN_ID
cfg['wandb_ckpt_run_path'] = WANDB_CKPT_RUN_PATH
cfg['wandb_ckpt_epoch'] = WANDB_CKPT_EPOCH
if WANDB_CKPT_CACHE_DIR is not None:
    cfg['wandb_ckpt_cache_dir'] = WANDB_CKPT_CACHE_DIR

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f)

print('Updated config:', config_path)

Updated config: /content/Hw5DiffusionTeam6/configs/ddpm.yaml


### 8) Enforce wandb Login (Fail Fast)
wandb is mandatory. The next cell will stop immediately if wandb is disabled or not authenticated.

In [15]:
import os
from pathlib import Path


def _load_env_file(path: Path):
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and value and key not in os.environ:
            os.environ[key] = value


env_path = Path(os.getcwd()) / ".env.local"
_load_env_file(env_path)

# Prefer Colab Secrets: add a secret named WANDB_API_KEY.
_api_key = os.environ.get("WANDB_API_KEY", "").strip()
if not _api_key:
    try:
        from google.colab import userdata

        _api_key = (userdata.get("WANDB_API_KEY") or "").strip()
    except Exception:
        pass
if not _api_key:
    raise RuntimeError(
        "Set WANDB_API_KEY in .env.local, Colab Secret WANDB_API_KEY, or the runtime environment before this cell. "
        "Do not commit real keys to Git."
    )
os.environ["WANDB_API_KEY"] = _api_key
print("WANDB_API_KEY loaded from environment.")

In [16]:
import os
import wandb

for k in ["WANDB_MODE", "WANDB_DISABLED"]:
    v = os.environ.get(k, "")
    if str(v).lower() in {"disabled", "true", "1", "offline"}:
        raise RuntimeError(f"Detected {k}={v}. Disabling/skipping wandb is not allowed. Stopping now.")

api_key = os.environ.get("WANDB_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("WANDB_API_KEY is missing. wandb is mandatory, so execution is stopped.")

try:
    wandb.login(key=api_key, relogin=True)
except Exception as e:
    raise RuntimeError(f"wandb login failed. Stopping now: {e}")

print("wandb login successful. Proceeding to training.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alexj2346 (alexj2346-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb login successful. Proceeding to training.


### 9) Start Training (Read Config from yaml Only)
Start training using `configs/ddpm.yaml` only.

In [ ]:
%cd /content/{REPO_NAME}
!python train.py --config configs/ddpm.yaml


/content/Hw5DiffusionTeam6
04/15/2026 21:53:34 - INFO - __main__ - Training with a single process on device cuda.
04/15/2026 21:53:34 - INFO - __main__ - Using dataset: /content/dataset_extracted/imagenet100_128x128/train
04/15/2026 21:53:34 - INFO - __main__ - Creating dataset
04/15/2026 21:53:34 - INFO - __main__ - Using subset: 39000/130000 samples (30.00%)
04/15/2026 21:53:34 - INFO - __main__ - Creating model
04/15/2026 21:53:36 - INFO - __main__ - UNet parameters: 78.99M
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
04/15/2026 21:53:42 - INFO - numexpr.utils - NumExpr defaulting to 12 threads.
Restored from /content/Hw5DiffusionTeam6/pretrained/model.ckpt
_IncompatibleKeys(missing_keys=[], unexpected_keys=['loss.logvar', 'loss.perceptual_loss.scaling_layer.shift', 'loss.perceptual_loss.scaling_layer.scale', 'loss.perceptual_loss.net.slice1.0.weight', 'loss.

### 10) Show Training Output Directory
Inspect output artifacts saved in Google Drive.

In [ ]:
OUTPUT_DIR = f"{DRIVE_ROOT}/experiments"
!ls -lah "$OUTPUT_DIR"